# Day 08 · 项目反应理论（二）：压缩评测集与自适应测验

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 按信息量挑选题目，并用分半验证检查挑选是否过拟合
- 模拟一次计算机自适应测验（CAT)，解释停止规则
- 说出把 CAT 用在大模型评测上的三个风险
- 判断什么时候单维 IRT 不适用

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 数据：120 个模型配置 × 60 道情景题

In [2]:
from course.data import simulate_vignette_matrix
from course.irt import fit_2pl, select_items_greedy, simulate_cat, total_information
u, theta, a_true, b_true, items = simulate_vignette_matrix(n_models=120, n_items=60, seed=8)
rng = np.random.default_rng(0)
perm = rng.permutation(120)
half_A, half_B = perm[:60], perm[60:]
fit_A = fit_2pl(u[half_A])
print("A 半拟合完成,迭代", fit_A["n_iter"])

A 半拟合完成,迭代 93


## 2. 在 A 半上挑题，在 B 半上检验

“全版分数”用 60 道题的通过率，“短版分数”用挑出的 k 道题的通过率。看两者在 B 半上的相关。

In [3]:
target = np.linspace(-1, 2, 31)       # 我们关心中等到强的模型
full_B = u[half_B].mean(1)
rows = []
for k in [5, 10, 15, 20, 30]:
    chosen = select_items_greedy(fit_A["a"], fit_A["b"], target, k)
    r_info = np.corrcoef(u[half_B][:, chosen].mean(1), full_B)[0, 1]
    r_rand = np.mean([np.corrcoef(u[half_B][:, rng.choice(60, k, replace=False)].mean(1), full_B)[0, 1] for _ in range(200)])
    rows.append({"k": k, "info-selected": r_info, "random": r_rand})
short = pd.DataFrame(rows); short

,k,info-selected,random
0,5,0.839,0.774
1,10,0.908,0.879
2,15,0.925,0.920
3,20,0.951,0.944
4,30,0.976,0.972


**练习 8.1**:k = 15 时，按信息量挑题比随机挑题的相关高多少？

In [4]:
ans_8_1 = None
check("练习 8.1", ans_8_1, float(short.set_index("k").loc[15, "info-selected"] - short.set_index("k").loc[15, "random"]), tol=0.005)

[ 待完成 ] 练习 8.1


**练习 8.2**：在 A 半上挑题、又在 A 半上检验，k = 15 时相关是多少？它比 B 半上的结果高还是低？填 `"higher"` 或 `"lower"`。

In [5]:
chosen15 = select_items_greedy(fit_A["a"], fit_A["b"], target, 15)
r_A = np.corrcoef(u[half_A][:, chosen15].mean(1), u[half_A].mean(1))[0, 1]
print("A 半自检相关:", round(r_A, 3))
ans_8_2 = None
check("练习 8.2", ans_8_2, "higher" if r_A > short.set_index("k").loc[15, "info-selected"] else "lower")

A 半自检相关: 0.967
[ 待完成 ] 练习 8.2


## 3. 自适应测验：每个模型需要几道题

In [6]:
steps = simulate_cat(true_theta=1.2, a=fit_A["a"], b=fit_A["b"], se_stop=0.35, max_items=40, seed=1)
pd.DataFrame(steps).round(3)

,item,response,theta_hat,se
0,35,1,0.668,0.711
1,17,0,0.240,0.509
2,1,1,0.458,0.472
3,21,0,0.207,0.415
4,15,1,0.272,0.383
5,18,1,0.356,0.374
6,46,0,0.258,0.347


**练习 8.3**：对 B 半的 60 个模型（用它们的真实 θ）各跑一次 CAT(se_stop=0.35, max_items=40)，平均用了几道题？

In [7]:
ans_8_3 = None
_lens = [len(simulate_cat(theta[i], fit_A["a"], fit_A["b"], se_stop=0.35, max_items=40, seed=i)) for i in half_B]
check("练习 8.3 平均题数", ans_8_3, float(np.mean(_lens)), tol=0.05)

[ 待完成 ] 练习 8.3 平均题数


## 4. 多维性：安全与共情不是同一个能力

模拟 120 个模型，30 道“安全”题由能力 θ₁ 驱动，30 道“共情”题由能力 θ₂ 驱动，两者相关只有 0.2。看题总相关。

In [8]:
from course.audit import item_analysis
rng2 = np.random.default_rng(3)
cov = [[1, .2], [.2, 1]]
th2 = rng2.multivariate_normal([0, 0], cov, 120)
b2 = rng2.normal(0, 1, 60)
dim = np.array([0] * 30 + [1] * 30)
p2 = 1 / (1 + np.exp(-1.5 * (th2[:, dim] - b2)))
u2 = (rng2.random(p2.shape) < p2).astype(float)
ia = item_analysis(u2)
ia["dimension"] = np.where(dim == 0, "safety", "empathy")
ia.groupby("dimension").item_rest_r.mean()

dimension
empathy    0.343
safety     0.368
Name: item_rest_r, dtype: float64

**练习 8.4**：分别计算“安全题内部”的题总相关均值（只用前 30 道题做 `item_analysis`）。它比上面混在一起时高还是低？填 `"higher"` 或 `"lower"`，并写一句话说明这对单维 IRT 意味着什么。

In [9]:
ans_8_4 = None
_within = item_analysis(u2[:, :30]).item_rest_r.mean()
_mixed = ia[ia.dimension == "safety"].item_rest_r.mean()
check("练习 8.4", ans_8_4, "higher" if _within > _mixed else "lower")

[ 待完成 ] 练习 8.4


*说明：*

## 口述题（录音，每题 90 秒）

1. 怎样用更少的题评测模型？
   <details><summary>参考要点</summary>估计题目参数后按信息量挑题，或做自适应测验；必须在独立的一组模型上检验短版效果；还要定期重估参数，因为新模型会改变题目结构，并注意题库曝光。</details>
2. 自适应测试在大模型评测里有什么风险？
   <details><summary>参考要点</summary>多维性、题目参数随模型代际漂移、题库曝光与污染、校准样本不足。</details>

## 研究延伸

一个可做的方向：用同一模型的多种系统提示（不同角色设定、不同 CBT 技术强调）扩充“被试”数量，估计心理健康情景题的难度和区分度，再做一个面向老年人场景的短版评测集。

In [10]:
summary()

本 notebook 自动批改:0 / 4 通过
